# Radiografía de Activaciones Token a Token: `CompactTrace` (INT4/Top-K) + `diff_traces` (Colab GPU)

Conecta las capturas token a token directamente con la geometría de `wsi` en tres pruebas reales (~2–4 min total en L4/A100):
1. **Loop onset & compresión real (`Qwen/Qwen3.5-0.8B`):** genera hasta 900 tokens en un problema de GSM8K, mide cuánto comprime `capture_compact` (`INT4` por micro-bloques de 16 + `Top-256` MLP + norma L2 por cabeza) frente a `trace_all` (`fp32`), mide fidelidad exacta (`cosine` / `rel_L2`) y localiza en qué token y capas arranca la repetición.
2. **Trigger plantado en 1 neurona (`Qwen/Qwen3.5-0.8B`):** inyecta un circuito en `(capa 20, neurona 42)` disparado por `" XYZ"` $\to$ `" Madrid"`, y verifica que `diff_traces(clean, trigger)` lo atrapa con nombre y apellido.
3. **Pushback vs. Neutral en Sycophancy (`Qwen/Qwen3.5-4B`):** radiografía exacta con `trace_all` sobre el último token del turno para ver qué capas, cabezas de atención y neuronas MLP cambian entre seguimiento neutral y presión del usuario.


In [ ]:
import os, shutil, json
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/wsi_runs/trace_xray"
OUT = "/content/results_trace_xray.json"
os.makedirs(DRIVE, exist_ok=True)


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!rm -rf /content/wsi-repo
!git clone --depth 1 -q https://github.com/krexito/weight-space-index.git /content/wsi-repo
%cd /content/wsi-repo
!pip install -q safetensors transformers accelerate


In [ ]:
!python -u scripts/experiment_trace_xray.py --out {OUT}
shutil.copy2(OUT, f"{DRIVE}/results_trace_xray.json")
d = json.load(open(OUT))
q8 = d["qwen35_08b"]
print("\n=== RESUMEN 0.8B ===")
print(f"tokens={q8['total_tokens']} | compact_full={q8['compact_full_bytes']/2**20:.2f} MB | ratio_32tok={q8['compression_ratio_32tok']}x")
print(f"INT4 cos={q8['int4_mean_cosine']} rel_L2={q8['int4_mean_rel_l2']} | INT8 cos={q8['int8_mean_cosine']} | MLP top-256 energy={q8['mlp_top256_energy_frac_mean']}")
print("loop onset:", q8["loop_onset"])
print("planted top MLP:", q8["planted_trigger_diff"]["top_mlp_contributors"][:3])
q4 = d["qwen35_4b_sycophancy_diff"]
print("\n=== RESUMEN 4B SYCOPHANCY DIFF ===")
print("neutral A..D:", q4["probs_neutral_ABCD"], "| pushback A..D:", q4["probs_pushback_ABCD"])
print("peak layer:", q4["peak_residual_layer"], "norm:", q4["peak_residual_delta_norm"])
print("top heads:", q4["top_attention_heads"][:5])
print("top MLPs:", q4["top_mlp_contributors"][:5])
from google.colab import files
files.download(OUT)


In [ ]:
# Apagar el runtime automáticamente si el resultado ya está guardado en Drive
AUTO_APAGAR = True
import time
if AUTO_APAGAR and os.path.exists(f"{DRIVE}/results_trace_xray.json"):
    time.sleep(60)
    from google.colab import runtime
    runtime.unassign()
